# K-means on Breast Cancer (Portfolio Sample)
This notebook demonstrates a clear, reproducible **unsupervised** workflow:
1) Load & scale data
2) Cluster with K-means
3) Evaluate with silhouette and label mapping
4) Visualize clusters in PCA space


In [ ]:

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, confusion_matrix, classification_report
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 1) Load data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")  # 0 = malignant, 1 = benign
print(X.shape, y.shape)


In [ ]:

# 2) Scale
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

# 3) K-means with k=2 (malignant/benign-like structure)
kmeans = KMeans(n_clusters=2, n_init=20, random_state=42)
labels = kmeans.fit_predict(Xs)

sil = silhouette_score(Xs, labels)
print("Silhouette Score:", round(sil, 3))


In [ ]:

# 4) Map cluster labels to true labels for interpretability
# We'll choose the mapping that maximizes accuracy.
def map_labels(pred, true):
    from itertools import permutations
    classes = np.unique(true)
    best_map, best_acc = None, -1
    for perm in permutations(range(len(np.unique(pred)))):
        mapped = np.array([perm[i] for i in pred])
        acc = (mapped == true).mean()
        if acc > best_acc:
            best_acc, best_map = acc, perm
    return best_map, best_acc

best_map, best_acc = map_labels(labels, y.values)
mapped = np.array([best_map[i] for i in labels])
print("Best accuracy after mapping:", round(best_acc, 3))

print(confusion_matrix(y, mapped))
print(classification_report(y, mapped, target_names=data.target_names))


In [ ]:

# 5) Visualization: PCA to 2D
pca = PCA(n_components=2, random_state=42)
Xp = pca.fit_transform(Xs)

plt.figure(figsize=(6,5))
for c in np.unique(mapped):
    plt.scatter(Xp[mapped==c, 0], Xp[mapped==c, 1], s=12, label=f'Cluster→{c}')
plt.xlabel('PCA 1'); plt.ylabel('PCA 2'); plt.legend(); plt.title('K-means clusters (mapped) in PCA space')
plt.show()


In [ ]:

# 6) K vs. silhouette (model selection cue)
sil_scores = []
K_range = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=20, random_state=42).fit(Xs)
    sil_scores.append(silhouette_score(Xs, km.labels_))

plt.figure(figsize=(6,4))
plt.plot(list(K_range), sil_scores, marker='o')
plt.xlabel('k'); plt.ylabel('Silhouette score'); plt.title('Silhouette vs. k')
plt.show()
